In [44]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KGramConstruction") \
    .getOrCreate()

sc = spark.sparkContext

26/03/01 18:15:46 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/01 18:15:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 18:15:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/01 18:15:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [45]:
import random
import hashlib
from itertools import combinations

In [46]:
base_path = "/home/rajesh/CSL7100/Assignment2/minhash/"

files = [
    base_path + "D1.txt",
    base_path + "D2.txt",
    base_path + "D3.txt",
    base_path + "D4.txt"
]

In [47]:
docs = {}

for i, file in enumerate(files):
    with open(file, "r") as f:
        docs[f"D{i+1}"] = f.read().strip()


In [48]:
print("D1: " + docs["D1"][:50])
print("D2: " + docs["D2"][:50])
print("D3: " + docs["D3"][:50])
print("D4: " + docs["D4"][:50])


D1: apple ceo tim cook is spending some time in canada
D2: apple ceo tim cook is spending some time in canada
D3: as part of his one day tour of canada yesterday ti
D4: president trump who warned as a candidate about th


## BUILD ALL k-GRAMS ONCE (Question 1)

In [49]:
def char_kgrams(text, k):
    text = text.strip()        # remove leading/trailing spaces
    result = set()             # use set to remove duplicates

    for i in range(len(text) - k + 1):   # valid start positions
        gram = text[i:i+k]               # substring of length k
        result.add(gram)                 # add to set (no duplicates)

    return result

In [50]:
def word_2grams(text):
    words = text.strip().split()   # split into words
    result = set()                 # use set to remove duplicates

    for i in range(len(words) - 1):   # until second-last word
        gram = words[i] + " " + words[i+1]   # combine two words
        result.add(gram)                   # add to set

    return result

In [51]:
char2_sets = {}
char3_sets = {}
word2_sets = {}

for name, text in docs.items():

    # Character 2-grams (set removes duplicates)
    char2_sets[name] = char_kgrams(text,2)

    # Character 3-grams
    char3_sets[name] = char_kgrams(text,3)

    # Word 2-grams
    words = text.split()
    word2_sets[name] = word_2grams(text)

In [52]:
print(list(char2_sets["D1"])[:5])
print(list(char3_sets["D2"])[:5])
print(list(word2_sets["D2"])[:5])

['cr', 'lt', 'er', 're', 'lo']
['tte', 'm c', 'd o', 'wee', 'ker']
['across canada', 'arkit cooks', 'welcome it', 'to differentiate', 'said ar']


# PART A: MINHASH USING SAVED 3-GRAMS

In [53]:
random.seed(42)                 # fix randomness
p = 2147483647                  # large prime

def stable_hash(x):
    return int(hashlib.md5(x.encode()).hexdigest(), 16)


In [54]:
set1 = char3_sets["D1"]         # reuse stored 3-grams
set2 = char3_sets["D2"]


In [55]:
for t in [20, 60, 150, 300, 600]:

    hash_functions = []
    for _ in range(t):
        a = random.randint(1, p-1)
        b = random.randint(0, p-1)
        hash_functions.append((a, b))

    sig1 = []
    sig2 = []

    for (a, b) in hash_functions:
        sig1.append(min((a * stable_hash(g) + b) % p for g in set1))
        sig2.append(min((a * stable_hash(g) + b) % p for g in set2))

    matches = sum(1 for i in range(t) if sig1[i] == sig2[i])

    print(f"t = {t}, Estimated Jaccard = {matches/t}")

print()

t = 20, Estimated Jaccard = 0.9
t = 60, Estimated Jaccard = 0.9666666666666667
t = 150, Estimated Jaccard = 0.9866666666666667
t = 300, Estimated Jaccard = 0.9733333333333334
t = 600, Estimated Jaccard = 0.98



# PART B: TRUE JACCARD USING SAVED k-GRAMS

In [56]:
from itertools import combinations

pairs = list(combinations(docs.keys(), 2))   # all document pairs (6 pairs for 4 docs)

for d1, d2 in pairs:

    # Convert to set just in case (prevents list error)
    A2 = set(char2_sets[d1])
    B2 = set(char2_sets[d2])

    A3 = set(char3_sets[d1])
    B3 = set(char3_sets[d2])

    W2A = set(word2_sets[d1])
    W2B = set(word2_sets[d2])

    # Jaccard similarity = |intersection| / |union|
    sim_char2 = len(A2 & B2) / len(A2 | B2)
    sim_char3 = len(A3 & B3) / len(A3 | B3)
    sim_word2 = len(W2A & W2B) / len(W2A | W2B)

    print(f"{d1} - {d2}")
    print("  Char 2-gram:", sim_char2)
    print("  Char 3-gram:", sim_char3)
    print("  Word 2-gram:", sim_word2)
    print()

D1 - D2
  Char 2-gram: 0.9811320754716981
  Char 3-gram: 0.977979274611399
  Word 2-gram: 0.9407665505226481

D1 - D3
  Char 2-gram: 0.8156996587030717
  Char 3-gram: 0.5803571428571429
  Word 2-gram: 0.18234165067178504

D1 - D4
  Char 2-gram: 0.6444444444444445
  Char 3-gram: 0.3050847457627119
  Word 2-gram: 0.03024193548387097

D2 - D3
  Char 2-gram: 0.8
  Char 3-gram: 0.5680473372781065
  Word 2-gram: 0.1736641221374046

D2 - D4
  Char 2-gram: 0.6412698412698413
  Char 3-gram: 0.30590339892665475
  Word 2-gram: 0.030303030303030304

D3 - D4
  Char 2-gram: 0.6529968454258676
  Char 3-gram: 0.31212381771281167
  Word 2-gram: 0.01607142857142857



In [58]:
if spark:
    spark.stop()